# SIAN: Sparse Interaction Additive Network

SIAN represents main effects and selected higher-order interactions in a block-masked ReLU network. It can discover interactions with Archipelago or accept an explicit sparse term set.


## Model


$$
\eta(x)=\beta_0+\sum_j f_j(x_j)+
\sum_{S\in\widehat{\mathcal I}} f_S(x_S),
$$

where $\widehat{\mathcal I}$ is selected from inclusion/removal contrasts of a reference network. Main effects are always retained.


## Shared estimator API

All neural estimators use `fit`, `predict`, `score`, `evaluate`, and
`predict_components`. The component result reconstructs predictions on the link
scale and supports shared term-importance and plotting utilities. Constructor
options such as `numerical_method` and `categorical_method` are forwarded to
PreTab and are fitted on training rows only.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(7)
n = 180
X = pd.DataFrame({
    "x1": rng.uniform(-1.0, 1.0, n),
    "x2": rng.normal(size=n),
    "group": rng.choice(["a", "b", "c"], size=n),
})
y = (
    np.sin(np.pi * X["x1"])
    + 0.35 * X["x2"] ** 2
    + 0.30 * (X["group"] == "b")
    + rng.normal(0.0, 0.12, n)
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=7
)

# Set True to run the small fit and all fitted-model demonstrations.
RUN_TRAINING = False


## Construct the estimator


In [ ]:
from nampy.models import SIANClassifier, SIANLSS, SIANRegressor


# Explicit terms bypass discovery and keep this example inexpensive.
model = SIANRegressor(
    interactions=(("x1", "x2"),),
    layer_sizes=[16, 12, 8],
    execution_mode="block_masked",
    l1_regularization=5e-5,
)
model.get_params(deep=False)


## Fit and inspect

Enable `RUN_TRAINING` above for a short demonstration. Real work should use a
larger validation set, enough epochs, and early stopping.


In [ ]:
if RUN_TRAINING:
    model.fit(
        X_train,
        y_train,
        max_epochs=3,
        batch_size=64,
        random_state=7,
        logger=False,
        enable_progress_bar=False,
        enable_model_summary=False,
    )
    predictions = model.predict(X_test)
    r2 = model.score(X_test, y_test)
    metrics = model.evaluate(X_test, y_test)
    components = model.predict_components(X_test, center=True)
    components.validate_additive_reconstruction()
    display({"R2": r2, **metrics})
    display(model.term_importance(X_test).head())


## Model-specific controls

Without explicit `interactions`, configure `max_interaction_order`, thresholds, and heredity. Fitted models can switch losslessly between block-masked and independent term execution.


In [ ]:
auto_model = SIANRegressor(
    max_interaction_order=2,
    interaction_thresholds=0.10,
    threshold_mode="fraction",
)
if RUN_TRAINING:
    display(model.selected_interactions_)
    model.compress_terms()
    compressed = model.predict(X_test)
    model.block_mask_terms()
    np.testing.assert_allclose(compressed, model.predict(X_test), rtol=1e-5, atol=1e-5)
    # For a fitted auto_model: auto_model.interaction_selection_table()


## Task variants and limits

`SIANRegressor`, `SIANClassifier`, and `SIANLSS` share interaction discovery and execution-mode controls.
